In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-OpenAI-API"

In [ ]:
#               DSX Data Scientist Coding Assessment
#-----------------------------------------------------------------------

# Question n.6: GenAI Clinical Data Assistant (LLM & LangChain)

# Datasets import
# CDISC SDTM AE domain dataset from pharmaversesdtm GitHub and the AE medidata dictionary

import pandas as pd
from Metadata_06_question import AE_METADATA

# GitHub raw URL
url = "https://raw.githubusercontent.com/pharmaverse/pharmaversesdtm/refs/heads/main/inst/extdata/ae.csv"

# AE dataset
adae = pd.read_csv(url)
# AE headers dictionary
ae_schema = AE_METADATA

In [ ]:
# Output definition
# Structured output containing target column and filter value
# Pydantic is used to define and validate the structure that we expect the LLM to return.

from pydantic import BaseModel, Field
from typing import List

# Output description
class FilterAE(BaseModel):
    target_column: str = Field(description="The AE dataset column to filter")
    filter_value: str = Field(description="The value to filter for")

# Multiple filters are needed to answer more articulate questions (multiple conditions)
class QueryStructure(BaseModel):
    filters: List[FilterAE] = Field(description="One or more filters that must be applied to the AE dataset")

In [ ]:
# LLM  -  LANGCHAIN
# CREATE THE CLINICAL TRIAL DATA AGENT

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

class ClinicalTrialDataAgent:

    def __init__(self, schema):
        
        # Store the AE metadata so it can be included in the prompt sent to the LLM.
        self.schema = schema
        
        # Create the connection to the LLM. temperature=0 makes the model more deterministic, which is useful for structured data-query generation.
        self.llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
        
        # Tell LangChain that we want the LLM response to conform to the Query Structure in Pydantic.
        self.structured_llm = self.llm.with_structured_output(QueryStructure)
        
        # LLM Prompt Engineering
        self.prompt = ChatPromptTemplate.from_messages([
            (
             "system",
        """
You are a clinical trial data assistant specializing in CDISC SDTM.
Your task is to translate a natural language question about an
Adverse Events (AE) dataset into one or more structured filters.

Use the following AE dataset schema:

{schema}

IMPORTANT RULES:
1. Identify EVERY independent filtering condition in the user's question.
2. If the question contains multiple conditions, return multiple filters.
3. Each filter must contain:
   - target_column
   - filter_value

4. Do NOT combine different concepts into a single filter.
5. Use the semantic meaning of the schema rather than exact
   keyword matching.

Examples:

Question:
"How many subjects had severe adverse events?"

Return:
[
    {{
        "target_column": "AESEV",
        "filter_value": "Severe"
    }}
]

Question:
"How many subjects experienced a severe headache?"

Return:
[
    {{
        "target_column": "AETERM",
        "filter_value": "Headache"
    }},
    {{
        "target_column": "AESEV",
        "filter_value": "Severe"
    }}
]

Question:
"How many subjects had cardiac serious adverse events?"

Return:
[
    {{
        "target_column": "AESOC",
        "filter_value": "Cardiac"
    }},
    {{
        "target_column": "AESER",
        "filter_value": "Y"
    }}
]

Return ONLY the structured output.
"""
    ),
    (
        "human",
        "{question}"
    )
])
    
    # TRANSLATE NATURAL LANGUAGE INTO A STRUCTURED QUERY
    def parse_question(self, question):

        # CHAIN
        # Combine the prompt with the structured LLM. 
        # The LLM receives: The AE metadata & The user's question and returns an Query Structure object.
        chain = self.prompt | self.structured_llm

        result = chain.invoke({
            "schema": self.schema,
            "question": question
        })

        return result
    
    # RUN THE COMPLETE PIPELINE
    def run(self, question, ae):
        
        # LLM to interpret the question
        query = self.parse_question(question)
        
        # Display the structured LLM output as JSON
        print("LLM QUERY:")
        print(query.model_dump_json(indent=2))

        result = execute_ae_query(ae, query)

        return result

In [ ]:
# EXECUTE THE STRUCTURED QUERY USING PANDAS
def execute_ae_query(ae, query: QueryStructure):
    
    # Start with the complete AE dataframe.
    filtered = ae.copy()
    
    # Apply every filter returned by the LLM.
    for f in query.filters:
        # Safety check: Make sure the LLM has selected a column that actually exists in the dataframe.
        if f.target_column not in ae.columns:
            raise ValueError(f"Column '{f.target_column}' does not exist.")
        
        # Apply the filter. 
        # strip() removes unnecessary spaces. 
        # lower() makes the comparison case-insensitive.
        filtered = filtered[
            filtered[f.target_column]
            .astype(str)
            .str.strip()
            .str.lower()
            == f.filter_value.strip().lower()
        ]

    # The question asks for UNIQUE patients/subjects. Therefore we use USUBJID and remove duplicates.
    subject_ids = (
        filtered["USUBJID"]
        .dropna()
        .unique()
        .tolist()
    )

    return {
        "subject_count": len(subject_ids),
        "subject_ids": subject_ids
    }

In [ ]:
# Example
agent = ClinicalTrialDataAgent(ae_schema)

result = agent.run(
    "How many unique patients experienced severe headache that was considered serious and related to the study treatment?",
    adae
)

print(result)

LLM QUERY:
{
  "filters": [
    {
      "target_column": "AETERM",
      "filter_value": "Headache"
    },
    {
      "target_column": "AESEV",
      "filter_value": "Severe"
    },
    {
      "target_column": "AESER",
      "filter_value": "Y"
    },
    {
      "target_column": "AEREL",
      "filter_value": "RELATED"
    }
  ]
}
{'subject_count': 0, 'subject_ids': []}
